In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1) Зареждане на dataset от Kaggle: чета CSV файла, който съдържа текста на имейлите и етикета (0=legit, 1=phishing).
PATH = "/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/phishing_email.csv"
df = pd.read_csv(PATH)

# Проверка на структурата: принтирам колони и примерни редове, за да съм сигурен как изглеждат данните.
print("Columns:", df.columns.tolist())
print(df.head(3))

# 2) Избор на колони: указвам коя колона е текстът на имейла и коя е етикетът (класът).
text_col = "text_combined"
label_col = "label"

# Подготовка на вход/изход: X е текстът (стринг), а y е бинарният етикет (0/1) за класификация.
X = df[text_col].astype(str).fillna("")
y = df[label_col].astype(int)  # 0=legit, 1=phishing

# 3) Разделяне train/test: отделям 20% за тест и запазвам баланса на класовете със stratify за коректна оценка.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) Модел (Pipeline): TF-IDF превръща текста в числови признаци, а Logistic Regression обучава бинарен класификатор с predict_proba().
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),       # ползвам думи и дву-думи фрази (n-grams), което помага при шаблони тип "verify account"
        max_features=50000,      # ограничавам признаците за по-бързо и стабилно обучение
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        solver="liblinear"
    ))
])

# Обучение: моделът се учи върху X_train и y_train (данни, които после няма да се ползват за тест).
model.fit(X_train, y_train)

# 5) Evaluation: оценявам модела върху test set (невиждани данни) чрез ROC-AUC и precision/recall/F1 метрики.
proba_test = model.predict_proba(X_test)[:, 1]     # вероятност за клас 1 (phishing)
pred_test = (proba_test >= 0.5).astype(int)        # предсказан клас при праг 0.5

print("\nROC-AUC:", roc_auc_score(y_test, proba_test))
print(classification_report(y_test, pred_test))

# 6) Sample email: дефинирам примерен имейл, който искам да оценя (входът от пощата).
sample_email = """From: stearnsupport@security.com
Subject: Urgent: Your account will be restricted within 24 hours
Dear customer,
We detected unusual activity on your account. To avoid temporary restriction, 
you must verify your identity immediately.
Please sign in using the secure link below:
http://stearnsupport-verification-login.secure-user-check.tz
If you do not verify your information within 24 hours, your access may be limited.
Thank you,
Security Team
"""

# 7) Phishing probability: взимам p за клас 1, което е confidence score на модела (оценка колко е вероятно да е phishing).
phishing_prob = model.predict_proba([sample_email])[0, 1]
print(f"\nPhishing probability: {phishing_prob:.3f}")

# 8) Explainability (local): извличам кои feature-и от конкретния имейл са допринесли най-много към резултата.
vec = model.named_steps["tfidf"]
clf = model.named_steps["clf"]

# Feature names и коефициенти: Logistic Regression има coef за всеки feature (положителен=бута към phishing, отрицателен=бута към legit).
feature_names = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]

# Векторизация на sample_email: получавам sparse TF-IDF вектор само за този имейл.
x = vec.transform([sample_email])
idx = x.indices                 # индексите на feature-ите, които реално присъстват в имейла
vals = x.data                   # TF-IDF стойностите за тези feature-и

# Принос (contribution): пресмятам локален принос като TF-IDF стойност * коефициент на модела (приближено обяснение "защо").
contrib = vals * coefs[idx]

# Топ feature-и, които увеличават phishing вероятността (най-голям положителен принос).
top_pos = np.argsort(contrib)[-15:][::-1]
print("\nTop contributing features (push to PHISH):")
for k in top_pos:
    i = idx[k]
    print(f"{feature_names[i]:<30} contrib={contrib[k]:.4f}  coef={coefs[i]:.4f}")

# Топ feature-и, които намаляват phishing вероятността (най-голям отрицателен принос).
top_neg = np.argsort(contrib)[:15]
print("\nTop contributing features (push to LEGIT):")
for k in top_neg:
    i = idx[k]
    print(f"{feature_names[i]:<30} contrib={contrib[k]:.4f}  coef={coefs[i]:.4f}")

Columns: ['text_combined', 'label']
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0

ROC-AUC: 0.99890712145636
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7919
           1       0.99      0.99      0.99      8579

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498


Phishing probability: 0.985

Top contributing features (push to PHISH):
account                        contrib=0.9184  coef=5.6133
http                           contrib=0.4511  coef=5.6948
secure                         contrib=0.3965  coef=1.8461
verify                         contrib=0.3563  coef=1.4256
within                         contrib=0.2901  coef=1.7053
se